# MOVIE RECOMMENDATION SYSTEM

### 1. Import required libraries

In [81]:
import numpy as np
import pandas as pd
import ast
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import nltk
from nltk.stem.porter import PorterStemmer

print("✅ Required libraries imported succesfully")
print()
print("✅ Numpy Imported: {}".format(np.__version__))
print("✅ Pandas Imported: {}".format(pd.__version__))
print("✅ AST Module imported")
print("✅ CountVectorizer imported")
print("✅ NLTK imported: {}".format(nltk.__version__))


✅ Required libraries imported succesfully

✅ Numpy Imported: 2.3.5
✅ Pandas Imported: 2.3.3
✅ AST Module imported
✅ CountVectorizer imported
✅ NLTK imported: 3.9.4


### 2. Import Dataset

In [19]:
movies = pd.read_csv("../Dataset/tmdb_5000_movies.csv")
credits = pd.read_csv("../Dataset/tmdb_5000_credits.csv")

movies.head(1)

,budget,genres,homepage,id,keywords,original_language,original_title,overview,popularity,production_companies,production_countries,release_date,revenue,runtime,spoken_languages,status,tagline,title,vote_average,vote_count
0,237000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://www.avatarmovie.com/,19995,"[{""id"": 1463, ""name"": ""culture clash""}, {""id"":...",en,Avatar,"In the 22nd century, a paraplegic Marine is di...",150.437577,"[{""name"": ""Ingenious Film Partners"", ""id"": 289...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2009-12-10,2787965087,162.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}, {""iso...",Released,Enter the World of Pandora.,Avatar,7.2,11800


In [20]:
credits.head(1)

,movie_id,title,cast,crew
0,19995,Avatar,"[{""cast_id"": 242, ""character"": ""Jake Sully"", ""...","[{""credit_id"": ""52fe48009251416c750aca23"", ""de..."


### 3. Merging both datasets

- movies = 20 columns
- credits = 4 columns

- new movies will have 23 columns 
- title will be added one time

In [21]:
movies  = movies.merge(credits, on='title')

In [22]:
movies.shape

(4809, 23)

In [23]:
movies.head(1)

,budget,genres,homepage,id,keywords,original_language,original_title,overview,popularity,production_companies,...,runtime,spoken_languages,status,tagline,title,vote_average,vote_count,movie_id,cast,crew
0,237000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://www.avatarmovie.com/,19995,"[{""id"": 1463, ""name"": ""culture clash""}, {""id"":...",en,Avatar,"In the 22nd century, a paraplegic Marine is di...",150.437577,"[{""name"": ""Ingenious Film Partners"", ""id"": 289...",...,162.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}, {""iso...",Released,Enter the World of Pandora.,Avatar,7.2,11800,19995,"[{""cast_id"": 242, ""character"": ""Jake Sully"", ""...","[{""credit_id"": ""52fe48009251416c750aca23"", ""de..."


### 4.  Droping Unwanted Columns

In [24]:
movies.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4809 entries, 0 to 4808
Data columns (total 23 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   budget                4809 non-null   int64  
 1   genres                4809 non-null   object 
 2   homepage              1713 non-null   object 
 3   id                    4809 non-null   int64  
 4   keywords              4809 non-null   object 
 5   original_language     4809 non-null   object 
 6   original_title        4809 non-null   object 
 7   overview              4806 non-null   object 
 8   popularity            4809 non-null   float64
 9   production_companies  4809 non-null   object 
 10  production_countries  4809 non-null   object 
 11  release_date          4808 non-null   object 
 12  revenue               4809 non-null   int64  
 13  runtime               4807 non-null   float64
 14  spoken_languages      4809 non-null   object 
 15  status               

In [25]:
# IMPORTANT COLUMNS

# genres
# id
# keywords
# title
# overview
# cast
# crew

In [26]:
movies = movies[['id', 'title', 'overview', 'genres', 'keywords', 'cast', 'crew']]

In [27]:
movies.head(1)

,id,title,overview,genres,keywords,cast,crew
0,19995,Avatar,"In the 22nd century, a paraplegic Marine is di...","[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...","[{""id"": 1463, ""name"": ""culture clash""}, {""id"":...","[{""cast_id"": 242, ""character"": ""Jake Sully"", ""...","[{""credit_id"": ""52fe48009251416c750aca23"", ""de..."


### `We will make a tag column by merging overview, genres, keywords, cast, crew all together`

### 5. Preprocessig

- #### Handling missing values

In [28]:
movies.isnull().sum()

id          0
title       0
overview    3
genres      0
keywords    0
cast        0
crew        0
dtype: int64

In [29]:
movies.dropna(inplace=True)

In [30]:
movies.isnull().sum()

id          0
title       0
overview    0
genres      0
keywords    0
cast        0
crew        0
dtype: int64

- #### Handling duplicate columns

In [31]:
print(movies.duplicated().sum())

0


- #### Converting genres column to proper format for merging
making a helper function for this issue

In [32]:
def convert(obj):
    L = []
    for i in ast.literal_eval(obj):
        L.append(i['name'])
    
    return L

In [33]:
movies['genres'] = movies['genres'].apply(convert)

In [34]:
movies.head(2)

,id,title,overview,genres,keywords,cast,crew
0,19995,Avatar,"In the 22nd century, a paraplegic Marine is di...","[Action, Adventure, Fantasy, Science Fiction]","[{""id"": 1463, ""name"": ""culture clash""}, {""id"":...","[{""cast_id"": 242, ""character"": ""Jake Sully"", ""...","[{""credit_id"": ""52fe48009251416c750aca23"", ""de..."
1,285,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha...","[Adventure, Fantasy, Action]","[{""id"": 270, ""name"": ""ocean""}, {""id"": 726, ""na...","[{""cast_id"": 4, ""character"": ""Captain Jack Spa...","[{""credit_id"": ""52fe4232c3a36847f800b579"", ""de..."


- #### Now keywords column to proper format for merging

In [35]:
movies['keywords'] = movies['keywords'].apply(convert)

In [36]:
movies.head(2)

,id,title,overview,genres,keywords,cast,crew
0,19995,Avatar,"In the 22nd century, a paraplegic Marine is di...","[Action, Adventure, Fantasy, Science Fiction]","[culture clash, future, space war, space colon...","[{""cast_id"": 242, ""character"": ""Jake Sully"", ""...","[{""credit_id"": ""52fe48009251416c750aca23"", ""de..."
1,285,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha...","[Adventure, Fantasy, Action]","[ocean, drug abuse, exotic island, east india ...","[{""cast_id"": 4, ""character"": ""Captain Jack Spa...","[{""credit_id"": ""52fe4232c3a36847f800b579"", ""de..."


- #### Now cast column to proper format for merging
- #### Now here we need first 3 actors names

In [37]:
def convert1(obj):
    L = []
    counter = 0
    for i in ast.literal_eval(obj):
        if counter != 3:
            L.append(i['name'])
            counter += 1
        else:
            break
    return L

In [38]:
movies['cast'] = movies['cast'].apply(convert1)

In [39]:
movies.head(2)

,id,title,overview,genres,keywords,cast,crew
0,19995,Avatar,"In the 22nd century, a paraplegic Marine is di...","[Action, Adventure, Fantasy, Science Fiction]","[culture clash, future, space war, space colon...","[Sam Worthington, Zoe Saldana, Sigourney Weaver]","[{""credit_id"": ""52fe48009251416c750aca23"", ""de..."
1,285,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha...","[Adventure, Fantasy, Action]","[ocean, drug abuse, exotic island, east india ...","[Johnny Depp, Orlando Bloom, Keira Knightley]","[{""credit_id"": ""52fe4232c3a36847f800b579"", ""de..."


- #### Now crew column to proper format
- #### We need name of crew where job is `Director`

In [40]:
def fetch_director(obj):
    L = []
    for i in ast.literal_eval(obj):
        if i['job'] == 'Director':
            L.append(i['name'])
            break
        else:
            continue
    return L

In [41]:
movies['crew'] = movies['crew'].apply(fetch_director)

- #### Converting overview column to proper format for merging

In [42]:
movies['overview'] = movies['overview'].apply(lambda x: x.split())

- #### Making genre, keywords, cast, crew values concatinate without spaces

In [43]:
movies['genres'] = movies['genres'].apply(lambda x:[i.replace(" ", '') for i in x])
movies['keywords'] = movies['keywords'].apply(lambda x:[i.replace(" ", '') for i in x])
movies['cast'] = movies['cast'].apply(lambda x:[i.replace(" ", '') for i in x])
movies['crew'] = movies['crew'].apply(lambda x:[i.replace(" ", '') for i in x])


In [44]:
movies.head(2)

,id,title,overview,genres,keywords,cast,crew
0,19995,Avatar,"[In, the, 22nd, century,, a, paraplegic, Marin...","[Action, Adventure, Fantasy, ScienceFiction]","[cultureclash, future, spacewar, spacecolony, ...","[SamWorthington, ZoeSaldana, SigourneyWeaver]",[JamesCameron]
1,285,Pirates of the Caribbean: At World's End,"[Captain, Barbossa,, long, believed, to, be, d...","[Adventure, Fantasy, Action]","[ocean, drugabuse, exoticisland, eastindiatrad...","[JohnnyDepp, OrlandoBloom, KeiraKnightley]",[GoreVerbinski]


- #### Making tags column

In [45]:
movies['tags'] = movies['overview'] + movies['keywords'] + movies['cast'] + movies['crew']

In [46]:
movies.head(2)

,id,title,overview,genres,keywords,cast,crew,tags
0,19995,Avatar,"[In, the, 22nd, century,, a, paraplegic, Marin...","[Action, Adventure, Fantasy, ScienceFiction]","[cultureclash, future, spacewar, spacecolony, ...","[SamWorthington, ZoeSaldana, SigourneyWeaver]",[JamesCameron],"[In, the, 22nd, century,, a, paraplegic, Marin..."
1,285,Pirates of the Caribbean: At World's End,"[Captain, Barbossa,, long, believed, to, be, d...","[Adventure, Fantasy, Action]","[ocean, drugabuse, exoticisland, eastindiatrad...","[JohnnyDepp, OrlandoBloom, KeiraKnightley]",[GoreVerbinski],"[Captain, Barbossa,, long, believed, to, be, d..."


In [47]:
new_movies = movies[['id', 'title', 'tags']]

In [48]:
new_movies.head(2)

,id,title,tags
0,19995,Avatar,"[In, the, 22nd, century,, a, paraplegic, Marin..."
1,285,Pirates of the Caribbean: At World's End,"[Captain, Barbossa,, long, believed, to, be, d..."


- #### Converting list of strings to strings in tags column

In [49]:
new_movies['tags'] = new_movies['tags'].apply(lambda x:" ".join(x))

C:\Users\admin\AppData\Local\Temp\ipykernel_7636\3966708295.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  new_movies['tags'] = new_movies['tags'].apply(lambda x:" ".join(x))


In [50]:
new_movies.head(2)

,id,title,tags
0,19995,Avatar,"In the 22nd century, a paraplegic Marine is di..."
1,285,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha..."


In [51]:
new_movies['tags'][0]

'In the 22nd century, a paraplegic Marine is dispatched to the moon Pandora on a unique mission, but becomes torn between following orders and protecting an alien civilization. cultureclash future spacewar spacecolony society spacetravel futuristic romance space alien tribe alienplanet cgi marine soldier battle loveaffair antiwar powerrelations mindandsoul 3d SamWorthington ZoeSaldana SigourneyWeaver JamesCameron'

### 6. Applying Natural Language Processing

- #### Applying Stemming
- We have an issue which is most words are repeted like (action, actions, activity, activities, ...)
- Converting All words to root words

In [67]:
ps = PorterStemmer()

In [68]:
def stem(text):
    y = []
    
    for i in text.split():
        y.append(ps.stem(i))
    
    return " ".join(y)

In [70]:
new_movies['tags'] = new_movies['tags'].apply(stem)

C:\Users\admin\AppData\Local\Temp\ipykernel_7636\3229886743.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  new_movies['tags'] = new_movies['tags'].apply(stem)


- #### Converting `tags` column to lowercase()

In [71]:
new_movies['tags'] = new_movies['tags'].apply(lambda x:x.lower())

C:\Users\admin\AppData\Local\Temp\ipykernel_7636\1627240426.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  new_movies['tags'] = new_movies['tags'].apply(lambda x:x.lower())


In [72]:
new_movies['tags'][0]

'in the 22nd century, a parapleg marin is dispatch to the moon pandora on a uniqu mission, but becom torn between follow order and protect an alien civilization. cultureclash futur spacewar spacecoloni societi spacetravel futurist romanc space alien tribe alienplanet cgi marin soldier battl loveaffair antiwar powerrel mindandsoul 3d samworthington zoesaldana sigourneyweav jamescameron'

### 7. Text Vectorization

Converting text to vectors is called Text Vectorization. It have many techniques like :
- TF IDF
- Bag of words
- word2vec
> But we will use `Bag of words`

In [73]:
cv = CountVectorizer(max_features=5000, stop_words='english')

In [74]:
vectors = cv.fit_transform(new_movies['tags']).toarray()

In [75]:
vectors

array([[0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       ...,
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0]], shape=(4806, 5000))

In [76]:
print(vectors[0])

[0 0 0 ... 0 0 0]


In [80]:
cv.get_feature_names_out()

array(['000', '007', '10', ..., 'zone', 'zoo', 'zooeydeschanel'],
      shape=(5000,), dtype=object)

#### 8. Now we will measure distance
- Not euclidian distance ( because it is not a relaible measure in higher dimensions )
- Instead we will use `consine distance` ( angle distance between vectors )

In [126]:
similarity = cosine_similarity(vectors)

In [127]:
similarity

array([[1.        , 0.        , 0.03184649, ..., 0.02475369, 0.        ,
        0.        ],
       [0.        , 1.        , 0.        , ..., 0.02592379, 0.        ,
        0.0277137 ],
       [0.03184649, 0.        , 1.        , ..., 0.02680281, 0.        ,
        0.        ],
       ...,
       [0.02475369, 0.02592379, 0.02680281, ..., 1.        , 0.0412393 ,
        0.04454354],
       [0.        , 0.        , 0.        , ..., 0.0412393 , 1.        ,
        0.08817334],
       [0.        , 0.0277137 , 0.        , ..., 0.04454354, 0.08817334,
        1.        ]], shape=(4806, 4806))

In [128]:
similarity.shape

(4806, 4806)

In [129]:
similarity[0]

array([1.        , 0.        , 0.03184649, ..., 0.02475369, 0.        ,
       0.        ], shape=(4806,))

### 9. Recommendation System

In [153]:
def recommend(movie):
    movie_index = new_movies[new_movies['title'] == movie].index[0]
    distance = similarity[movie_index]
    movies_list = sorted(list(enumerate(distance)), reverse=True, key = lambda x:x[1])[1:6]

    for i in movies_list:
        print(new_movies.iloc[i[0]].title)

In [192]:
recommend("Blade: Trinity")

Blade
Blade II
Dracula 2000
Underworld: Evolution
Dracula


In [189]:
new_movies.sample(10)

,id,title,tags
4686,54702,Malevolence,it' ten year after the kidnap of martin bristo...
3238,11217,Club Dread,when a serial killer interrupt the fun at the ...
101,49538,X-Men: First Class,befor charl xavier and erik lensherr took the ...
2419,9792,The Hills Have Eyes,"base on we craven' 1977 suspens cult classic, ..."
156,616,The Last Samurai,nathan algren is an american hire to instruct ...
4287,223,Rebecca,a self-consci bride is torment by the memori o...
654,36648,Blade: Trinity,"for years, blade ha fought against the vampir ..."
2464,10781,The Texas Chainsaw Massacre: The Beginning,chrissi and her friend set out on a road trip ...
4404,320146,Creative Control,smooth advertis execut david is in a relations...
2937,31909,Invaders from Mars,"in thi remak of the classic 50 sf tale, a boy ..."
